# Task 5 — Mental Health Support Chatbot (Fine-Tuned DistilGPT2)

**Goal:** Fine-tune `distilgpt2` on **EmpatheticDialogues** so the model learns **gentler, more supportive** language for stress and emotional wellness (educational demo — not a substitute for therapy).

**Hardware:** Training is much faster on a **GPU** (e.g. Colab T4, local CUDA). CPU works but is slower; reduce `max_samples` and `batch_size` if needed.


## Theory: loss function and what we optimize

DistilGPT2 is a **causal language model**: it predicts the **next token** given all previous tokens.

- **Loss:** **cross-entropy** over the vocabulary at each position (standard **negative log-likelihood** for next-token prediction).
- **Objective:** Minimize this loss so the model assigns higher probability to **empathetic continuations** that appear in EmpatheticDialogues after our `### Situation:` / `### Supportive reply:` template.
- **Why not “empathy loss”?** There is no separate empathy metric in the Trainer; *empathy* emerges from **imitation** of human listeners in the dataset.

`Trainer` + `DataCollatorForLanguageModeling(..., mlm=False)` sets `labels = input_ids` so padding positions are typically ignored when `pad_token_id` is set (we align pad with EOS for DistilGPT2).


## 0. Install dependencies (run once)


In [6]:

# pip install datasets transformers accelerate evaluate torch safetensors
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "transformers", "accelerate", "evaluate", "torch", "safetensors"])


0

## 1. Imports and configuration


In [7]:

import os
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)

# Output directory for tokenizer + weights (used by app.py)
OUTPUT_DIR = Path("./mental_health_distilgpt2")  # noqa: S108
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Limit training size for quick runs (set None for full training — slower)
MAX_TRAIN_SAMPLES = 8000
MAX_EVAL_SAMPLES = 800

BASE_MODEL = "distilgpt2"


## 2. Load **EmpatheticDialogues** (parquet revision)

The Hugging Face Hub hosts a **parquet** conversion of this dataset (script-based loaders are deprecated in newer `datasets` versions). We use `revision="refs/convert/parquet"`.


In [8]:

def clean_text(t: str) -> str:
    """Dataset uses placeholders like _comma_ in text fields."""
    return (
        t.replace("_comma_", ",")
        .replace("_period_", ".")
        .replace("_exclamation_", "!")
        .replace("_question_", "?")
    )


raw = load_dataset(
    "facebook/empathetic_dialogues",
    revision="refs/convert/parquet",
)
train_raw = raw["train"]
eval_raw = raw["validation"]

if MAX_TRAIN_SAMPLES is not None:
    train_raw = train_raw.select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
if MAX_EVAL_SAMPLES is not None:
    eval_raw = eval_raw.select(range(min(MAX_EVAL_SAMPLES, len(eval_raw))))


def to_training_example(batch):
    prompts = [clean_text(p) for p in batch["prompt"]]
    utts = [clean_text(u) for u in batch["utterance"]]
    texts = [
        f"### Situation:\n{p}\n### Supportive reply:\n{u}"
        for p, u in zip(prompts, utts)
    ]
    return {"text": texts}


train_ds = train_raw.map(
    to_training_example,
    batched=True,
    remove_columns=train_raw.column_names,
)
eval_ds = eval_raw.map(
    to_training_example,
    batched=True,
    remove_columns=eval_raw.column_names,
)

train_ds[0]


{'text': '### Situation:\nI remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.\n### Supportive reply:\nI remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.'}

## 3. Tokenizer and model


In [9]:

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)


def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )


tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_train


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 8000
})

## 4. Fine-tune with `Trainer` + `TrainingArguments`

- **`fp16`:** mixed precision on GPU (faster, less memory).
- **`gradient_accumulation_steps`:** effective larger batch on small GPUs.


In [10]:

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

trainer.train()


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.412001,2.667399


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1000, training_loss=2.618501182556152, metrics={'train_runtime': 212.1862, 'train_samples_per_second': 37.703, 'train_steps_per_second': 4.713, 'total_flos': 522593501184000.0, 'train_loss': 2.618501182556152, 'epoch': 1.0})

## 5. Save tokenizer + fine-tuned weights


In [11]:

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved to:", OUTPUT_DIR.resolve())


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/mental_health_distilgpt2


## 6. Quick sanity check: generation


In [12]:

from transformers import pipeline

gen = pipeline(
    "text-generation",
    model=str(OUTPUT_DIR),
    tokenizer=str(OUTPUT_DIR),
    device=0 if torch.cuda.is_available() else -1,
)
prompt = (
    "### Situation:\n"
    "I have been feeling really stressed about exams and I cannot focus.\n"
    "### Supportive reply:\n"
)
print(gen(prompt, max_new_tokens=80, do_sample=True, temperature=0.75, top_p=0.9)[0]["generated_text"])


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Situation:
I have been feeling really stressed about exams and I cannot focus.
### Supportive reply:
I am so stressed out about exams and I can't focus. I am always stressed out about exams and I cannot focus. I am always stressed out about exams and I cannot focus. I am always stressed out about exams and I cannot focus. I am always stressed out about exams and I cannot focus. I am always stressed out about exams and I cannot focus. I am always stressed out about exams
